# Critic Agent DPO Training

Fine-tune **Qwen 3.5 9B** with Direct Preference Optimization (DPO) to improve the critic agent's ability to detect hallucinations in clinical answers.

**Training Data:** ~570 preference pairs
- 70 pairs from our EHR evaluation (real clinical data)
- 500 pairs from MedHallu dataset (medical hallucination benchmark)

**Requirements:** Google Colab Pro (T4 16GB or A100 40GB)

## Steps
1. Install dependencies
2. Clone repo & load data
3. Generate DPO preference pairs (local EHR + MedHallu)
4. Train with DPO + QLoRA
5. Evaluate before/after
6. Export model

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers trl peft datasets bitsandbytes accelerate
!pip install -q sentencepiece protobuf

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.0f} GB")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > GPU")

## 2. Clone Repo & Load Data

In [ ]:
import os

# Clone from GitHub
if not os.path.exists("ehr-copilot"):
    !git clone https://github.com/shas232/SJSU-MSE.git ehr-copilot

os.chdir("ehr-copilot")
print(f"Working directory: {os.getcwd()}")
!ls data/ results/ scripts/

## 3. Generate DPO Pairs

This downloads the **MedHallu** dataset (10K medical hallucination examples from UT Austin, used at EMNLP 2025) and combines with our local EHR eval pairs.

In [ ]:
import json
from pathlib import Path

# Always regenerate to include MedHallu
print("Generating DPO pairs (local EHR + MedHallu)...")
!python scripts/generate_dpo_pairs.py --medhallu_count 500

# Verify
data_path = Path("data/dpo_pairs_hf.jsonl")
with open(data_path) as f:
    pairs = [json.loads(l) for l in f]
print(f"\nTotal pairs: {len(pairs)}")
print(f"Sample prompt length: {len(pairs[0]['prompt'])} chars")

## 4. Load Model + Dataset

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen3.5-9B"

# Load dataset
records = []
with open("data/dpo_pairs_hf.jsonl") as f:
    for line in f:
        records.append(json.loads(line.strip()))

dataset = Dataset.from_list(records)
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])}, Eval: {len(split['test'])}")

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load Qwen 3.5 9B
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

mem_gb = torch.cuda.memory_allocated() / 1024**3
print(f"Qwen 3.5 9B loaded. GPU memory: {mem_gb:.1f} GB")

## 5. Train with DPO

In [ ]:
from trl import DPOConfig, DPOTrainer

# LoRA config — target all attention + MLP projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

# Training config
training_args = DPOConfig(
    output_dir="models/critic-dpo",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size = 8
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    max_length=2048,
    max_prompt_length=1536,
    beta=0.1,  # DPO temperature
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

# DPO Trainer
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # TRL handles ref model with peft
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Starting DPO training...")
print(f"  Base model:       Qwen 3.5 9B")
print(f"  Train samples:    {len(split['train'])}")
print(f"  Eval samples:     {len(split['test'])}")
print(f"  Effective batch:  {2 * 4}")
print(f"  Epochs:           3")
print(f"  LoRA rank:        16")
print(f"  DPO beta:         0.1")

In [ ]:
# Run training (~15 min on T4, ~5 min on A100)
trainer.train()

In [ ]:
# Save final model
trainer.save_model("models/critic-dpo/final")
tokenizer.save_pretrained("models/critic-dpo/final")
print("Model saved to models/critic-dpo/final/")

# Print training metrics
metrics = trainer.state.log_history
train_losses = [m["loss"] for m in metrics if "loss" in m]
eval_losses = [m["eval_loss"] for m in metrics if "eval_loss" in m]
if train_losses:
    print(f"  Final train loss: {train_losses[-1]:.4f}")
if eval_losses:
    print(f"  Final eval loss:  {eval_losses[-1]:.4f}")

## 6. Evaluate: Verdict Accuracy

In [ ]:
import json
import re

test_pairs = [split["test"][i] for i in range(len(split["test"]))]
print(f"Evaluating on {len(test_pairs)} test pairs...")

def generate_critic_response(model, tokenizer, prompt, max_new_tokens=512):
    """Generate a critic response from the model."""
    # Qwen 3.5 uses thinking mode by default; disable for structured output
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,  # direct mode for JSON output
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response


def parse_verdict(response):
    """Extract verdict from model response."""
    try:
        json_match = re.search(r'\{[^{}]*"verdict"[^{}]*\}', response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            return parsed.get("verdict", "UNKNOWN").upper()
    except (json.JSONDecodeError, AttributeError):
        pass
    upper = response.upper()
    for v in ["APPROVED", "REVISED", "ABSTAINED"]:
        if v in upper:
            return v
    return "UNKNOWN"


def get_expected_verdict(chosen_text):
    try:
        return json.loads(chosen_text).get("verdict", "UNKNOWN").upper()
    except json.JSONDecodeError:
        return "UNKNOWN"

In [ ]:
# Run evaluation
correct = 0
total = 0
confusion = {"APPROVED": {"APPROVED": 0, "REVISED": 0, "ABSTAINED": 0, "UNKNOWN": 0},
             "REVISED": {"APPROVED": 0, "REVISED": 0, "ABSTAINED": 0, "UNKNOWN": 0},
             "ABSTAINED": {"APPROVED": 0, "REVISED": 0, "ABSTAINED": 0, "UNKNOWN": 0}}

for pair in test_pairs:
    expected = get_expected_verdict(pair["chosen"])
    response = generate_critic_response(model, tokenizer, pair["prompt"])
    predicted = parse_verdict(response)
    
    match = predicted == expected
    if match:
        correct += 1
    total += 1
    
    if expected in confusion and predicted in confusion[expected]:
        confusion[expected][predicted] += 1
    
    print(f"  [{total}] Expected: {expected:10s} | Predicted: {predicted:10s} | {'OK' if match else 'MISS'}")

accuracy = correct / total if total > 0 else 0
print(f"\n{'=' * 50}")
print(f"DPO Critic Verdict Accuracy: {accuracy:.1%} ({correct}/{total})")
print(f"{'=' * 50}")

# Confusion matrix
print(f"\nConfusion Matrix (rows=expected, cols=predicted):")
print(f"{'':>12} {'APPROVED':>10} {'REVISED':>10} {'ABSTAINED':>10} {'UNKNOWN':>10}")
for exp in ["APPROVED", "REVISED", "ABSTAINED"]:
    row = f"{exp:>12}"
    for pred in ["APPROVED", "REVISED", "ABSTAINED", "UNKNOWN"]:
        row += f" {confusion[exp][pred]:>10}"
    print(row)

## 7. Export Model

In [ ]:
# Download LoRA adapter (~50MB zip)
!zip -r critic-dpo-lora.zip models/critic-dpo/final/

from google.colab import files
files.download("critic-dpo-lora.zip")
print("\nDownloaded! To use in production:")
print("  from peft import PeftModel")
print("  base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3.5-9B')")
print("  model = PeftModel.from_pretrained(base, 'models/critic-dpo/final')")

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment to merge LoRA + push

# from peft import PeftModel
# from huggingface_hub import login
#
# login()  # Paste your HF token
#
# base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
# merged = PeftModel.from_pretrained(base, "models/critic-dpo/final")
# merged = merged.merge_and_unload()
#
# merged.push_to_hub("shas232/ehr-critic-dpo-qwen3.5-9b")
# tokenizer.push_to_hub("shas232/ehr-critic-dpo-qwen3.5-9b")
# print("Pushed to HuggingFace Hub!")

## 8. Training Summary

| Component | Details |
|-----------|----------|
| Base Model | **Qwen 3.5 9B** (Apache 2.0, released Mar 2026) |
| Method | DPO (Direct Preference Optimization) |
| Quantization | QLoRA (4-bit NF4) |
| LoRA Rank | 16 (alpha=32) |
| Training Data | **~570 preference pairs** |
| — Local EHR | 70 pairs from 10-patient MIMIC-IV evaluation |
| — MedHallu | 500 pairs (UT Austin medical hallucination benchmark) |
| Pair Types | Grounded→Approve vs False-Abstain, Hallucinated→Revise vs False-Approve |
| Epochs | 3 |
| DPO Beta | 0.1 |
| Learning Rate | 5e-5 (cosine decay) |
| Hardware | Google Colab Pro (T4 16GB / A100 40GB) |